# Chapter 78 - Special Tokens

Chapter 77 turned ordered BPE-style merge rules into a tokenizer that could train, encode, decode, and restore its state.

This chapter adds optional tokens whose roles come from the way a dataset or generation loop uses them:

```text
<unknown>
<start>
<end>
```

The model receives only integer IDs, so a token becomes special through our data conventions rather than through its spelling.

## Learning goals

By the end of this chapter, you will be able to:

- explain why a special token is still an ordinary vocabulary entry to a model;
- configure optional unknown, start, and end tokens;
- distinguish lossless known-text encoding from lossy unknown-token fallback;
- add and remove boundary markers without confusing them with ordinary text;
- connect tokenizer vocabulary size to model embedding and output sizes;
- explain how an end token can stop generation; and
- explain why fixed-length GPT slices do not need padding yet.

## Terms and contracts

A **special token** is a vocabulary entry whose role is defined by a data-processing convention.

An **unknown token** replaces input that the tokenizer cannot otherwise represent.

A **start token** and an **end token** mark sequence boundaries.

A **padding token** fills unused positions when variable-length sequences must share a rectangular batch.

The tokenizer in this chapter uses these contracts:

- known text still decodes exactly;
- each unknown character becomes one visible `<unknown>` token;
- start and end markers are added structurally, not recognized by parsing their spellings from ordinary text;
- decoding hides start and end markers by default but keeps `<unknown>` visible; and
- special-token configuration is saved as part of tokenizer state.

## Special to us, ordinary to the model

Suppose a tokenizer assigns IDs to `<start>`, `<end>`, and `the`.

A GPT receives only those integer IDs.

It can learn that the start-token ID often begins an example only if training examples consistently place that ID at the beginning.

Adding a token to the vocabulary without using it in training does not teach the model its intended meaning.

Every added token also needs one embedding row and one output score, so tokenizer and model vocabulary sizes must match exactly.

## Design choices for the teaching tokenizer

Only requested special tokens are added.

Their spellings are reserved and cannot also be encoded as ordinary user text, which prevents a literal string such as `"<end>"` from being confused with a generated boundary marker.

Unknown fallback happens before learned merges are replayed.

That makes `encode_to_subwords()` honest: it shows `<unknown>` exactly where `encode()` will emit the unknown-token ID.

This remains character-initialized BPE-style tokenization rather than production byte-level BPE.

## Core types and merge helpers

The first cell retains Chapter 77's typed merge rule, deterministic lexical tie-break, and non-overlapping merge scan.

The state format now also records the three feature flags and the configured special-token list.

In [1]:
from collections import Counter
from dataclasses import dataclass
from typing import Self

TokenPair = tuple[str, str]

UNKNOWN_TOKEN = "<unknown>"
START_TOKEN = "<start>"
END_TOKEN = "<end>"


@dataclass(frozen=True)
class MergeRule:
    index: int
    pair: TokenPair
    new_token: str
    pair_count: int
    replacement_count: int


@dataclass(frozen=True)
class TokenizerState:
    version: int
    number_of_merges: int
    use_unknown_token: bool
    use_start_token: bool
    use_end_token: bool
    merge_rules: tuple[MergeRule, ...]
    base_alphabet: tuple[str, ...]
    special_tokens: tuple[str, ...]
    subword_id_items: tuple[tuple[str, int], ...]


def build_special_token_list(
    use_unknown_token: bool,
    use_start_token: bool,
    use_end_token: bool,
) -> list[str]:
    special_tokens: list[str] = []

    if use_unknown_token:
        special_tokens.append(UNKNOWN_TOKEN)
    if use_start_token:
        special_tokens.append(START_TOKEN)
    if use_end_token:
        special_tokens.append(END_TOKEN)

    return special_tokens


def count_adjacent_token_pairs(
    token_sequence: list[str],
) -> Counter[TokenPair]:
    return Counter(zip(token_sequence, token_sequence[1:], strict=False))


def select_most_frequent_pair(
    pair_counts: Counter[TokenPair],
) -> tuple[TokenPair, int]:
    if not pair_counts:
        raise ValueError("No adjacent token pairs are available.")

    highest_count = max(pair_counts.values())
    tied_pairs = [
        pair
        for pair, count in pair_counts.items()
        if count == highest_count
    ]
    return min(tied_pairs), highest_count


def merge_pair_in_sequence(
    token_sequence: list[str],
    pair_to_merge: TokenPair,
    merged_token: str,
) -> tuple[list[str], int]:
    merged_sequence: list[str] = []
    replacement_count = 0
    position = 0

    while position < len(token_sequence):
        pair_matches = (
            position + 1 < len(token_sequence)
            and token_sequence[position] == pair_to_merge[0]
            and token_sequence[position + 1] == pair_to_merge[1]
        )

        if pair_matches:
            merged_sequence.append(merged_token)
            replacement_count += 1
            position += 2
        else:
            merged_sequence.append(token_sequence[position])
            position += 1

    return merged_sequence, replacement_count


def apply_merge_rules(
    token_sequence: list[str],
    merge_rules: list[MergeRule],
) -> list[str]:
    merged_sequence = list(token_sequence)

    for merge_rule in merge_rules:
        merged_sequence, _ = merge_pair_in_sequence(
            token_sequence=merged_sequence,
            pair_to_merge=merge_rule.pair,
            merged_token=merge_rule.new_token,
        )

    return merged_sequence

`build_special_token_list()` creates a stable configuration order, while vocabulary IDs will be assigned by sorting the complete vocabulary.

The merge algorithm itself does not need special-token logic because special tokens are never part of merge training.

## The enhanced tokenizer

The constructor decides which special tokens exist before training.

Changing that choice later would change the ID mapping, so a model should be created only after the final tokenizer configuration has been trained.

As in Chapter 77, `train()` computes new state locally and commits it only after all checks pass.

In [2]:
class HomemadeSubwordTokenizer:
    number_of_merges: int
    use_unknown_token: bool
    use_start_token: bool
    use_end_token: bool
    merge_rules: list[MergeRule]
    base_alphabet: set[str]
    special_tokens: tuple[str, ...]
    subword_to_id: dict[str, int]
    id_to_subword: dict[int, str]
    is_trained: bool

    def __init__(
        self,
        number_of_merges: int,
        use_unknown_token: bool = False,
        use_start_token: bool = False,
        use_end_token: bool = False,
    ) -> None:
        if number_of_merges < 0:
            raise ValueError("number_of_merges must be nonnegative.")

        self.number_of_merges = number_of_merges
        self.use_unknown_token = use_unknown_token
        self.use_start_token = use_start_token
        self.use_end_token = use_end_token
        self.merge_rules = []
        self.base_alphabet = set()
        self.special_tokens = tuple(
            build_special_token_list(
                use_unknown_token=use_unknown_token,
                use_start_token=use_start_token,
                use_end_token=use_end_token,
            )
        )
        self.subword_to_id = {}
        self.id_to_subword = {}
        self.is_trained = False

    @property
    def vocabulary_size(self) -> int:
        self._check_trained()
        return len(self.subword_to_id)

    def _check_trained(self) -> None:
        if not self.is_trained:
            raise RuntimeError("Tokenizer must be trained before use.")

    def _check_reserved_spellings(self, text: str) -> None:
        present_spellings = [
            special_token
            for special_token in self.special_tokens
            if special_token in text
        ]
        if present_spellings:
            raise ValueError(
                "Ordinary text contains reserved special-token spellings: "
                f"{present_spellings!r}."
            )

    def _validate_internal_state(self) -> None:
        configured_tokens = tuple(
            build_special_token_list(
                use_unknown_token=self.use_unknown_token,
                use_start_token=self.use_start_token,
                use_end_token=self.use_end_token,
            )
        )
        if self.special_tokens != configured_tokens:
            raise ValueError("Special tokens do not match the feature flags.")

        expected_ids = set(range(len(self.subword_to_id)))
        if set(self.subword_to_id.values()) != expected_ids:
            raise ValueError("Vocabulary IDs must be unique and contiguous from zero.")

        expected_inverse = {
            token_id: subword
            for subword, token_id in self.subword_to_id.items()
        }
        if self.id_to_subword != expected_inverse:
            raise ValueError("Vocabulary mappings are not inverses.")

        vocabulary = set(self.subword_to_id)
        required_tokens = (
            self.base_alphabet
            | set(self.special_tokens)
            | {rule.new_token for rule in self.merge_rules}
        )
        if not required_tokens.issubset(vocabulary):
            raise ValueError("The vocabulary is missing required tokens.")
        if len(self.merge_rules) > self.number_of_merges:
            raise ValueError("The state contains too many merge rules.")

        for expected_index, rule in enumerate(self.merge_rules):
            if rule.index != expected_index:
                raise ValueError("Merge-rule indices must be consecutive.")
            if rule.new_token != "".join(rule.pair):
                raise ValueError("A learned token must concatenate its pair.")
            if rule.pair_count < 1 or rule.replacement_count < 1:
                raise ValueError("Merge counts must be positive.")
            if rule.new_token in self.special_tokens:
                raise ValueError("A learned token collides with a special token.")

    def train(self, text: str) -> None:
        if not text:
            raise ValueError("Training text must not be empty.")

        self._check_reserved_spellings(text)
        token_sequence = list(text)
        base_alphabet = set(token_sequence)
        vocabulary = set(base_alphabet)
        merge_rules: list[MergeRule] = []

        for merge_index in range(self.number_of_merges):
            pair_counts = count_adjacent_token_pairs(token_sequence)
            if not pair_counts:
                break

            pair_to_merge, pair_count = select_most_frequent_pair(pair_counts)
            new_token = "".join(pair_to_merge)
            if new_token in vocabulary or new_token in self.special_tokens:
                raise RuntimeError(f"Merge token collision for {new_token!r}.")

            token_sequence, replacement_count = merge_pair_in_sequence(
                token_sequence=token_sequence,
                pair_to_merge=pair_to_merge,
                merged_token=new_token,
            )
            merge_rules.append(
                MergeRule(
                    index=merge_index,
                    pair=pair_to_merge,
                    new_token=new_token,
                    pair_count=pair_count,
                    replacement_count=replacement_count,
                )
            )
            vocabulary.add(new_token)

        vocabulary.update(self.special_tokens)
        sorted_vocabulary = sorted(vocabulary)
        subword_to_id = {
            subword: token_id
            for token_id, subword in enumerate(sorted_vocabulary)
        }
        id_to_subword = {
            token_id: subword
            for subword, token_id in subword_to_id.items()
        }

        self.merge_rules = merge_rules
        self.base_alphabet = base_alphabet
        self.subword_to_id = subword_to_id
        self.id_to_subword = id_to_subword
        self.is_trained = True
        self._validate_internal_state()

    def encode_to_subwords(
        self,
        text: str,
        add_boundaries: bool = True,
    ) -> list[str]:
        self._check_trained()
        self._check_reserved_spellings(text)

        ordinary_tokens: list[str] = []
        unknown_characters: set[str] = set()
        for character in text:
            if character in self.base_alphabet:
                ordinary_tokens.append(character)
            elif self.use_unknown_token:
                ordinary_tokens.append(UNKNOWN_TOKEN)
                unknown_characters.add(character)
            else:
                unknown_characters.add(character)

        if unknown_characters and not self.use_unknown_token:
            raise ValueError(
                "Text contains characters not seen during training: "
                f"{sorted(unknown_characters)!r}."
            )

        subwords = apply_merge_rules(ordinary_tokens, self.merge_rules)
        if add_boundaries and self.use_start_token:
            subwords.insert(0, START_TOKEN)
        if add_boundaries and self.use_end_token:
            subwords.append(END_TOKEN)
        return subwords

    def encode(
        self,
        text: str,
        add_boundaries: bool = True,
    ) -> list[int]:
        subwords = self.encode_to_subwords(
            text=text,
            add_boundaries=add_boundaries,
        )
        return [self.subword_to_id[subword] for subword in subwords]

    def decode(
        self,
        token_ids: list[int],
        skip_boundary_tokens: bool = True,
    ) -> str:
        self._check_trained()
        invalid_token_ids = sorted(set(token_ids) - set(self.id_to_subword))
        if invalid_token_ids:
            raise ValueError(f"Unknown token IDs: {invalid_token_ids!r}.")

        boundary_tokens = {START_TOKEN, END_TOKEN}
        subwords = [self.id_to_subword[token_id] for token_id in token_ids]
        if skip_boundary_tokens:
            subwords = [
                subword
                for subword in subwords
                if subword not in boundary_tokens
            ]
        return "".join(subwords)

    def get_state(self) -> TokenizerState:
        self._check_trained()
        subword_id_items = tuple(
            sorted(self.subword_to_id.items(), key=lambda item: item[1])
        )
        return TokenizerState(
            version=2,
            number_of_merges=self.number_of_merges,
            use_unknown_token=self.use_unknown_token,
            use_start_token=self.use_start_token,
            use_end_token=self.use_end_token,
            merge_rules=tuple(self.merge_rules),
            base_alphabet=tuple(sorted(self.base_alphabet)),
            special_tokens=self.special_tokens,
            subword_id_items=subword_id_items,
        )

    @classmethod
    def from_state(cls, state: TokenizerState) -> Self:
        if state.version != 2:
            raise ValueError(f"Unsupported tokenizer-state version {state.version}.")

        tokenizer = cls(
            number_of_merges=state.number_of_merges,
            use_unknown_token=state.use_unknown_token,
            use_start_token=state.use_start_token,
            use_end_token=state.use_end_token,
        )
        tokenizer.merge_rules = list(state.merge_rules)
        tokenizer.base_alphabet = set(state.base_alphabet)
        tokenizer.special_tokens = state.special_tokens
        tokenizer.subword_to_id = dict(state.subword_id_items)
        tokenizer.id_to_subword = {
            token_id: subword
            for subword, token_id in tokenizer.subword_to_id.items()
        }
        tokenizer.is_trained = True
        tokenizer._validate_internal_state()
        return tokenizer

The tokenizer never searches ordinary input for marker syntax.

Callers request boundaries through `add_boundaries`, which keeps data text separate from control tokens.

The default decoder removes only `<start>` and `<end>`; an unknown marker remains visible because silently deleting missing content would make debugging harder.

## Add only the tokens that are requested

The same training text and merge count produce the same learned ordinary tokens in each tokenizer below.

Only their configured special-token entries differ.

In [3]:
training_text = "the cat sat on the mat. the cat sat."

tokenizer_without_special_tokens = HomemadeSubwordTokenizer(
    number_of_merges=20,
)
tokenizer_with_boundaries = HomemadeSubwordTokenizer(
    number_of_merges=20,
    use_start_token=True,
    use_end_token=True,
)
tokenizer_with_all_three = HomemadeSubwordTokenizer(
    number_of_merges=20,
    use_unknown_token=True,
    use_start_token=True,
    use_end_token=True,
)

configured_tokenizers = [
    tokenizer_without_special_tokens,
    tokenizer_with_boundaries,
    tokenizer_with_all_three,
]
for configured_tokenizer in configured_tokenizers:
    configured_tokenizer.train(training_text)

print("configuration       | special tokens                    | vocabulary")
print("-" * 76)
configuration_names = ["none", "start and end", "all three"]
for configuration_name, configured_tokenizer in zip(
    configuration_names,
    configured_tokenizers,
    strict=True,
):
    print(
        f"{configuration_name:<19} | "
        f"{str(configured_tokenizer.special_tokens):<33} | "
        f"{configured_tokenizer.vocabulary_size}"
    )

assert (
    tokenizer_with_all_three.vocabulary_size
    == tokenizer_without_special_tokens.vocabulary_size + 3
)

configuration       | special tokens                    | vocabulary
----------------------------------------------------------------------------
none                | ()                                | 30
start and end       | ('<start>', '<end>')              | 32
all three           | ('<unknown>', '<start>', '<end>') | 33


The vocabulary grows by exactly the number of configured special tokens.

Because IDs are assigned from the final sorted vocabulary, adding special tokens can also change existing ordinary-token IDs.

Never add tokens after a model has already learned embeddings for an older mapping.

## Start and end boundaries

The next example adds markers around known text, decodes ordinary user-facing text, and optionally exposes the markers for inspection.

In [4]:
boundary_text = "the cat sat"
boundary_subwords = tokenizer_with_boundaries.encode_to_subwords(boundary_text)
boundary_token_ids = tokenizer_with_boundaries.encode(boundary_text)
decoded_without_boundaries = tokenizer_with_boundaries.decode(boundary_token_ids)
decoded_with_boundaries = tokenizer_with_boundaries.decode(
    boundary_token_ids,
    skip_boundary_tokens=False,
)

print("Original:", repr(boundary_text))
print("Subwords:", boundary_subwords)
print("Token IDs:", boundary_token_ids)
print("User-facing decode:", repr(decoded_without_boundaries))
print("Debug decode:", repr(decoded_with_boundaries))

assert boundary_subwords[0] == START_TOKEN
assert boundary_subwords[-1] == END_TOKEN
assert decoded_without_boundaries == boundary_text

Original: 'the cat sat'
Subwords: ['<start>', 'the ', 'cat s', 'at', '<end>']
Token IDs: [9, 30, 19, 11, 8]
User-facing decode: 'the cat sat'
Debug decode: '<start>the cat sat<end>'


Skipping the boundary IDs restores the original known text exactly.

Keeping them visible is a debugging view rather than an exact text round trip.

Boundaries can also be disabled for a particular encoding when processing one continuous stream rather than separate records.

In [5]:
without_boundary_ids = tokenizer_with_boundaries.encode(
    boundary_text,
    add_boundaries=False,
)

print("With boundaries:", boundary_token_ids)
print("Without boundaries:", without_boundary_ids)
print(
    "Boundary IDs:",
    tokenizer_with_boundaries.subword_to_id[START_TOKEN],
    tokenizer_with_boundaries.subword_to_id[END_TOKEN],
)

assert len(boundary_token_ids) == len(without_boundary_ids) + 2
assert tokenizer_with_boundaries.decode(without_boundary_ids) == boundary_text

With boundaries: [9, 30, 19, 11, 8]
Without boundaries: [30, 19, 11]
Boundary IDs: 9 8


For separate documents, boundaries help only when every training document is encoded with the same convention.

For fixed slices of one continuous text stream, adding boundaries to every arbitrary slice would falsely label slice edges as document boundaries.

## Unknown fallback is deliberately lossy

The training text contains lowercase letters but not uppercase `T` or the question mark.

Without an unknown token, encoding such characters raises a clear error.

With fallback enabled, each unseen character becomes one `<unknown>` token before merge rules are applied.

Because this tokenizer retains every base character and learned token, a missing merged subword would indicate inconsistent tokenizer state rather than a normal fallback case.

In [6]:
unknown_text = "The cat?"

try:
    tokenizer_without_special_tokens.encode(unknown_text)
except ValueError as error:
    print("Without fallback:", error)

unknown_subwords = tokenizer_with_all_three.encode_to_subwords(unknown_text)
unknown_token_ids = tokenizer_with_all_three.encode(unknown_text)
unknown_decoded = tokenizer_with_all_three.decode(unknown_token_ids)

print()
print("Original:", repr(unknown_text))
print("Subwords:", unknown_subwords)
print("Token IDs:", unknown_token_ids)
print("Decoded:", repr(unknown_decoded))
print("Exact round trip:", unknown_decoded == unknown_text)

assert unknown_subwords.count(UNKNOWN_TOKEN) == 2
assert unknown_decoded == "<unknown>he cat<unknown>"
assert unknown_decoded != unknown_text

Without fallback: Text contains characters not seen during training: ['?', 'T'].

Original: 'The cat?'
Subwords: ['<start>', '<unknown>', 'he ', 'c', 'at', '<unknown>', '<end>']
Token IDs: [9, 10, 25, 19, 12, 10, 8]
Decoded: '<unknown>he cat<unknown>'
Exact round trip: False


The visible markers show both lost characters, but they cannot reveal whether those characters were `T`, `?`, or something else.

Known text still round-trips exactly even when unknown fallback is available.

In [7]:
known_text = "the cat sat"
known_token_ids = tokenizer_with_all_three.encode(known_text)
known_decoded = tokenizer_with_all_three.decode(known_token_ids)

print("Known IDs:", known_token_ids)
print("Known decode:", repr(known_decoded))

assert known_decoded == known_text

Known IDs: [9, 31, 20, 12, 8]
Known decode: 'the cat sat'


`<unknown>` is not removed by the default decoder.

Removing it would hide the fact that information was lost and could join surrounding fragments misleadingly.

Production byte-level tokenizers often avoid this problem by representing arbitrary input bytes instead of falling back to one lossy unknown token.

## Reserved spellings prevent ambiguity

Special markers are control tokens, not text syntax in this implementation.

The tokenizer therefore rejects their literal spellings in training data and ordinary input.

In [8]:
try:
    tokenizer_with_boundaries.encode("the <end> cat")
except ValueError as error:
    print("Reserved input:", error)

try:
    invalid_training_tokenizer = HomemadeSubwordTokenizer(
        number_of_merges=4,
        use_end_token=True,
    )
    invalid_training_tokenizer.train("a literal <end> marker")
except ValueError as error:
    print("Reserved training text:", error)

Reserved input: Ordinary text contains reserved special-token spellings: ['<end>'].
Reserved training text: Ordinary text contains reserved special-token spellings: ['<end>'].


Without this restriction, a literal marker string and a structurally inserted marker could become indistinguishable after tokenization.

A production tokenizer can support literal marker text, but it needs an explicit escaping or allowed-special-token policy.

## End tokens and generation

An end token can define a stopping condition for autoregressive generation.

The loop must compare the sampled ID with the tokenizer's actual end-token ID rather than assume a fixed number.

In [9]:
def should_stop_generation(
    sampled_token_id: int,
    tokenizer: HomemadeSubwordTokenizer,
) -> bool:
    if not tokenizer.use_end_token:
        return False

    end_token_id = tokenizer.subword_to_id[END_TOKEN]
    return sampled_token_id == end_token_id


end_token_id = tokenizer_with_boundaries.subword_to_id[END_TOKEN]
ordinary_token_id = tokenizer_with_boundaries.encode(
    "the",
    add_boundaries=False,
)[0]

print("End-token ID:", end_token_id)
print(
    "Stop for ordinary token:",
    should_stop_generation(ordinary_token_id, tokenizer_with_boundaries),
)
print(
    "Stop for end token:",
    should_stop_generation(end_token_id, tokenizer_with_boundaries),
)

assert not should_stop_generation(ordinary_token_id, tokenizer_with_boundaries)
assert should_stop_generation(end_token_id, tokenizer_with_boundaries)

End-token ID: 8
Stop for ordinary token: False
Stop for end token: True


This stopping behavior is useful only after the model has seen end-token targets during training.

Adding `<end>` to the vocabulary without placing it at real sequence endings gives the model no examples from which to learn that convention.

## Vocabulary size is a model contract

The largest valid token ID must be `vocabulary_size - 1` because IDs are contiguous from zero.

A GPT created for this tokenizer needs exactly this many embedding rows and output scores.

In [10]:
model_vocabulary_size = tokenizer_with_all_three.vocabulary_size
maximum_token_id = max(tokenizer_with_all_three.id_to_subword)

print("Tokenizer vocabulary size:", model_vocabulary_size)
print("Maximum token ID:", maximum_token_id)
print("Special-token IDs:")
for special_token in tokenizer_with_all_three.special_tokens:
    print(
        f"  {special_token:<10} -> "
        f"{tokenizer_with_all_three.subword_to_id[special_token]}"
    )

assert maximum_token_id == model_vocabulary_size - 1
assert len(tokenizer_with_all_three.subword_to_id) == model_vocabulary_size

Tokenizer vocabulary size: 33
Maximum token ID: 32
Special-token IDs:
  <unknown>  -> 10
  <start>    -> 9
  <end>      -> 8


The later model construction must use `tokenizer.vocabulary_size` rather than an earlier or manually guessed value.

A vocabulary that is too small causes out-of-range IDs, while a larger one creates output classes that this tokenizer can never emit.

## Special-token state must survive restoration

The snapshot includes the feature flags, marker list, merge rules, alphabet, and exact ID assignments.

Restoring all of them keeps model-facing token meanings unchanged.

In [11]:
tokenizer_state = tokenizer_with_all_three.get_state()
restored_tokenizer = HomemadeSubwordTokenizer.from_state(tokenizer_state)

state_test_text = "the cat"
original_ids = tokenizer_with_all_three.encode(state_test_text)
restored_ids = restored_tokenizer.encode(state_test_text)

print("State version:", tokenizer_state.version)
print("Stored special tokens:", tokenizer_state.special_tokens)
print("Original IDs:", original_ids)
print("Restored IDs:", restored_ids)

assert restored_tokenizer.get_state() == tokenizer_state
assert restored_ids == original_ids
assert restored_tokenizer.decode(restored_ids) == state_test_text

State version: 2
Stored special tokens: ('<unknown>', '<start>', '<end>')
Original IDs: [9, 31, 19, 12, 8]
Restored IDs: [9, 31, 19, 12, 8]


## Why there is no padding token yet

Current GPT batches are fixed-length slices from one long token stream.

Every row already has the same context length, so padding would not solve a current problem.

Padding becomes useful when variable-length records must share one rectangular batch.

That later design also needs:

- an attention policy for padded positions;
- a loss policy that ignores padded targets; and
- a clear decision about whether padding belongs on the left or right.

Adding `<padding>` now would increase the vocabulary without implementing any of those required behaviors.

## Common mistakes

- Do not assume marker spelling gives a token meaning to the model.
- Do not add special tokens after training a model with an older ID mapping.
- Do not silently delete unknown markers during decoding.
- Do not expect unknown-token fallback to preserve unseen characters.
- Do not insert boundaries at arbitrary fixed-length slice edges and call them document boundaries.
- Do not stop on a guessed end-token ID.
- Do not add padding without attention and loss policies for padded positions.
- Do not use a model checkpoint with a different tokenizer state.

## Takeaways

Special tokens are ordinary vocabulary entries whose roles come from consistent use in data and generation code.

This chapter added optional support for:

- `<unknown>` as a visible but lossy fallback;
- `<start>` as a sequence-beginning marker; and
- `<end>` as a sequence-ending and generation-stopping marker.

Known text can still round-trip exactly when boundary markers are skipped.

Unknown text cannot round-trip exactly after distinct characters collapse to the same unknown ID.

Every configured special token changes the tokenizer vocabulary and therefore the model's required `vocabulary_size` and ID interpretation.

## What comes next

The next chapter can encode a training corpus with this tokenizer and build shifted GPT input-target batches from the resulting subword IDs.

That step will make boundary placement a dataset decision and will use the tokenizer's final vocabulary size when constructing the model.